In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
%pip install faiss-cpu

In [3]:
from google.colab import drive
import sys
import os
# %pip install -q faiss-gpu-cu12
%pip install -q duckdb polars

sys.path.append(os.path.abspath('/content/drive/My Drive/Thesis/book_recsys'))
sys.path.insert(0,'/content/drive/My Drive/Thesis/book_recsys/app')

from app.recommenders.vector_recommender import VectorRecommender
from app.recommenders.processer import BookProcessor, UserProcessor

In [4]:
_base_dir = '/content/drive/My Drive/Thesis/book_recsys'
_data_dir = f'{_base_dir}/data/processed_2'
_models_dir = f'{_data_dir}/model_v1'
_eval_dir = f'{_data_dir}/eval_v1'

user_index = f'{_models_dir}/user_index.csv'
imtem_index = f'{_models_dir}/cb_sbert_book_index.csv'
history_db = f'{_models_dir}/user_history.db'

sbert_item_matrix = f'{_models_dir}/cb_sbert_matrix.npy'
sbert_user_matrix = f'{_models_dir}/cb_sbert_user_matrix.npy'
sbert_item_faiss_index = f'{_models_dir}/cb_sbert_matrix.index'

ials_user_matrix = f'{_models_dir}/cf_als_user_profiles.npy'
ials_item_matrix = f'{_models_dir}/cf_als_item_matrix.npy'
ials_item_faiss_index = f'{_models_dir}/cf_als_item_matrix.index'

test_file = f'{_data_dir}/test_main.csv'
ground_truth_file = f'{_eval_dir}/test_ground_truth.json'

cbf_predictions = f'{_eval_dir}/cbf_prediction.json'
cf_predictions = f'{_eval_dir}/cf_test_predictions.json'

cbf_predictions_v2 = f'{_eval_dir}/cbf_test_predictions_v2.json'
cbf_prediction_v2_w1 = f'{_eval_dir}/cbf_test_predictions_v2_w1.json'
cbf_prediction_v2_w2 = f'{_eval_dir}/cbf_test_predictions_v2_w2.json'


In [ ]:
# import pandas as pd
# import json
# import os

# # Tạo thư mục output nếu chưa có
# os.makedirs("output", exist_ok=True)

# print("Đang đọc tập Test...")
# # ĐIỀN ĐÚNG ĐƯỜNG DẪN TỚI FILE TEST CỦA BẠN (VD: test_interactions.csv)
# test_df = pd.read_csv(test_file)

# # Ép kiểu user_id và book_id về chuỗi (String) để đồng bộ với toàn hệ thống
# test_df['user_id'] = test_df['user_id'].astype(str)
# test_df['book_id'] = test_df['book_id'].astype(str)

# print("Đang gộp nhóm sách theo từng User...")
# # Group by user id
# ground_truth_dict = test_df.groupby('user_id')['book_id'].apply(list).to_dict()

# # save on disk
# gt_path = f"{_eval_dir}/test_ground_truth.json"
# with open(gt_path, 'w', encoding='utf-8') as f:
#     json.dump(ground_truth_dict, f, ensure_ascii=False, indent=2)

# print(f"Ground Truth saved:  {len(ground_truth_dict):,} users at: {gt_path}")

# # Trích xuất danh sách user_id để dùng cho Cell 2 và Cell 3
# test_users_list = list(ground_truth_dict.keys())

Đang đọc tập Test...
Đang gộp nhóm sách theo từng User...
Ground Truth saved:  743,401 users at: /content/drive/My Drive/Thesis/book_recsys/data/processed_2/eval_v1/test_ground_truth.json


In [ ]:
import json
import time
import gc
# from processer import UserProcessor, BookProcessor
# from recommender import UniversalVectorRecommender

def get_chunks(lst, chunk_size=10_000):
    for i in range(0, len(lst), chunk_size):
        yield lst[i:i + chunk_size]

def append_predictions(predictions_dict, output_path, is_first_write=False):

    if not predictions_dict:
        return

    json_ready_data = {
        str(uid): [item.to_api_dict() for item in rec_items]
        for uid, rec_items in predictions_dict.items()
    }

    # Format từng key-value thành chuỗi chuẩn JSON
    items_str = []
    for k, v in json_ready_data.items():
        items_str.append(f'  "{k}": {json.dumps(v, ensure_ascii=False)}')

    with open(output_path, 'a', encoding='utf-8') as f:
        if not is_first_write:
            f.write(",\n")
        f.write(",\n".join(items_str))

def main():
    print("Start prediction")
    try:
      del cf_engine
      del cbf_engine
      del user_proc
      del book_proc
    except:
      pass
    gc.collect()
    # ==========================================
    # 1. Load user List
    # ==========================================
    gt_path = ground_truth
    with open(gt_path, 'r', encoding='utf-8') as f:
        ground_truth_data = json.load(f)
    test_users = list(ground_truth_data.keys())
    print(f"loading {len(test_users):,} users from Test.")

    # ==========================================
    # 2. Initializting
    # ==========================================
    print("-------Data Processors...")
    book_proc = BookProcessor(item_index_path= imtem_index)
    user_proc = UserProcessor(user_index_path= user_index, history_db_path= history_db)
    gc.collect()
    chunks = list(get_chunks(test_users, 5000))

    # ==========================================
    # 3.  CF (ALS)
    # ==========================================
    print("\n" + "="*40 + "\nRunning CF (ALS)...")
    cf_engine = VectorRecommender(
        book_processor= book_proc,
        user_processor= user_proc,
        faiss_index_path=ials_item_faiss_index,
        user_vectors_path = ials_user_matrix,
    )


    start_time = time.time()
    cf_output_path = f"{_eval_dir}/cf_test_predictions.json"

    # Khởi tạo file JSON với dấu mở ngoặc
    with open(cf_output_path, 'w', encoding='utf-8') as f:
        f.write("{\n")

    is_first_write = True
    for i, chunk in enumerate(chunks, 1):
        print(f" -> Processing {i}/{len(chunks)}...", end="\r")
        preds = cf_engine.recommend_batch(chunk, k=50)

        if preds:
            append_predictions(preds, cf_output_path, is_first_write)
            is_first_write = False

    # Đóng file JSON
    with open(cf_output_path, 'a', encoding='utf-8') as f:
        f.write("\n}\n")

    print(f"\nCF Done {time.time() - start_time:.2f} second.")
    del cf_engine
    gc.collect()

    i = 32
    while i <=128:
      print(f" -> Processing {i}/{128}...", end="\r")
      cf_engine = VectorRecommender(
        book_processor= book_proc,
        user_processor= user_proc,
        faiss_index_path=f'{_data_dir}/model_v3/cf_als_item_matrix__{i}.index',
        user_vectors_path = f'{_data_dir}/model_v3/cf_als_user_profiles_{i}.npy',
      )

      start_time = time.time()
      cf_output_path = f"{_eval_dir}/cf_test_predictions_{i}.json"

      # Khởi tạo file JSON với dấu mở ngoặc
      with open(cf_output_path, 'w', encoding='utf-8') as f:
          f.write("{\n")

      is_first_write = True
      for i, chunk in enumerate(chunks, 1):
          print(f" -> Processing {i}/{len(chunks)}...", end="\r")
          preds = cf_engine.recommend_batch(chunk, k=50)

          if preds:
              append_predictions(preds, cf_output_path, is_first_write)
              is_first_write = False

      # Đóng file JSON
      with open(cf_output_path, 'a', encoding='utf-8') as f:
          f.write("\n}\n")

      print(f"\nCF Done {time.time() - start_time:.2f} second.")
      del cf_engine
      gc.collect()
      i=i*2

    # ==========================================
    # 4. CBF (NLP)
    # ==========================================
    print("="*40 + "\nRunning CBF (NLP)...")
    cbf_engine = VectorRecommender(
        book_processor= book_proc,
        user_processor= user_proc,
        faiss_index_path=sbert_item_faiss_index,
        user_vectors_path = sbert_user_matrix,
    )

    start_time = time.time()
    cbf_output_path = f"{_eval_dir}/cbf_test_predictions.json"

    # Khởi tạo file JSON với dấu mở ngoặc
    with open(cbf_output_path, 'w', encoding='utf-8') as f:
        f.write("{\n")

    is_first_write = True
    for i, chunk in enumerate(chunks, 1):
        print(f" ---- Processing {i}/{len(chunks)}...", end="\r")
        preds = cbf_engine.recommend_batc h(chunk, k=50)

        if preds:
            append_predictions(preds, cbf_output_path, is_first_write)
            is_first_write = False

    # Đóng file JSON
    with open(cbf_output_path, 'a', encoding='utf-8') as f:
        f.write("\n}\n")

    print(f"\nDone {time.time() - start_time:.2f} giây.")
    print("ALL DONE")

# if __name__ == "__main__":
#     main()


In [8]:
import json
import time
import os

def parse_json_line(line):
    line = line.strip()
    if line in ["{", "}"] or not line:
        return None, None
    if line.endswith(','):
        line = line[:-1]
    json_str = "{" + line + "}"
    try:
        data = json.loads(json_str)
        uid = list(data.keys())[0]
        items = data[uid]
        return uid, items
    except Exception:
        return None, None

def normalize_scores(items):
    if not items:
        return {}
    scores = [item['score'] for item in items]
    min_s, max_s = min(scores), max(scores)
    norm_dict = {}
    for item in items:
        if max_s == min_s:
            norm_score = 1.0
        else:
            norm_score = (item['score'] - min_s) / (max_s - min_s)
        norm_dict[item['item_id']] = norm_score
    return norm_dict

def run_hybrid(cf_path, cbf_path, output_path, weight_cf, weight_cbf, k_top=10):
    start_time = time.time()
    processed_users = 0

    with open(cf_path, 'r', encoding='utf-8') as f_cf, \
         open(cbf_path, 'r', encoding='utf-8') as f_cbf, \
         open(output_path, 'w', encoding='utf-8') as f_out:

        f_out.write("{\n")

        for line_cf, line_cbf in zip(f_cf, f_cbf):
            uid_cf, items_cf = parse_json_line(line_cf)
            uid_cbf, items_cbf = parse_json_line(line_cbf)

            if not uid_cf or not uid_cbf:
                continue
            if uid_cf != uid_cbf:
                print(f"WARNING: user mismatch {uid_cf} vs {uid_cbf}, skipping")
                continue

            cf_norm = normalize_scores(items_cf)
            cbf_norm = normalize_scores(items_cbf)
            all_item_ids = set(cf_norm.keys()).union(set(cbf_norm.keys()))

            hybrid_scores = []
            for item_id in all_item_ids:
                s_cf = cf_norm.get(item_id, 0.0)
                s_cbf = cbf_norm.get(item_id, 0.0)
                final_score = (s_cf * weight_cf) + (s_cbf * weight_cbf)
                hybrid_scores.append({"item_id": item_id, "score": final_score})

            hybrid_scores.sort(key=lambda x: x['score'], reverse=True)
            top_k = hybrid_scores[:k_top]

            for rank, item in enumerate(top_k, start=1):
                item['rank'] = rank

            json_record = f'  "{uid_cf}": {json.dumps(top_k, ensure_ascii=False)}'
            if processed_users > 0:
                f_out.write(",\n" + json_record)
            else:
                f_out.write(json_record)

            processed_users += 1
            if processed_users % 10000 == 0:
                print(f"  [{weight_cf:.1f}/{weight_cbf:.1f}] processed {processed_users:,} users", end="\r")

        f_out.write("\n}")

    elapsed = time.time() - start_time
    print(f"  [{weight_cf:.1f}/{weight_cbf:.1f}] done: {processed_users:,} users in {elapsed:.2f}s -> {output_path}")
    return processed_users

def main():
    cf_path = cf_predictions      # adjust path
    cbf_path = cbf_prediction_v2_w1   # adjust path
    output_dir = f"{_eval_dir}/hybrid"
    os.makedirs(output_dir, exist_ok=True)

    weight_pairs = [
        (1.0, 0.0),
        (0.8, 0.2),
        (0.6, 0.4),
        (0.4, 0.6),
        (0.2, 0.8),
        (0.0, 1.0),
    ]

    print(f"Running hybrid fusion for {len(weight_pairs)} weight configurations\n")

    for w_cf, w_cbf in weight_pairs:
        filename = f"hybrid_{w_cf:.1f}_{w_cbf:.1f}_v2_w1.json"
        output_path = os.path.join(output_dir, filename)
        run_hybrid(cf_path, cbf_path, output_path, w_cf, w_cbf)

    print(f"\nAll {len(weight_pairs)} configurations complete. Files saved in '{output_dir}/'")

if __name__ == "__main__":
    main()

Running hybrid fusion for 6 weight configurations

  [1.0/0.0] done: 743,401 users in 254.22s -> /content/drive/My Drive/Thesis/book_recsys/data/processed_2/eval_v1/hybrid/hybrid_1.0_0.0_v2_w1.json
  [0.8/0.2] done: 743,401 users in 252.43s -> /content/drive/My Drive/Thesis/book_recsys/data/processed_2/eval_v1/hybrid/hybrid_0.8_0.2_v2_w1.json
  [0.6/0.4] done: 743,401 users in 265.55s -> /content/drive/My Drive/Thesis/book_recsys/data/processed_2/eval_v1/hybrid/hybrid_0.6_0.4_v2_w1.json
  [0.4/0.6] done: 743,401 users in 258.03s -> /content/drive/My Drive/Thesis/book_recsys/data/processed_2/eval_v1/hybrid/hybrid_0.4_0.6_v2_w1.json
  [0.2/0.8] done: 743,401 users in 254.42s -> /content/drive/My Drive/Thesis/book_recsys/data/processed_2/eval_v1/hybrid/hybrid_0.2_0.8_v2_w1.json
  [0.0/1.0] done: 743,401 users in 268.16s -> /content/drive/My Drive/Thesis/book_recsys/data/processed_2/eval_v1/hybrid/hybrid_0.0_1.0_v2_w1.json

All 6 configurations complete. Files saved in '/content/drive/My D

Start prediction
loading 743,401 users from Test.
-------Data Processors...
Loading Book mapping from /content/drive/My Drive/Thesis/book_recsys/data/processed_2/model_v1/cb_sbert_book_index.csv ...
 -> Loaded 1,173,713 books into RAM dictionary.
Loading User mapping from /content/drive/My Drive/Thesis/book_recsys/data/processed_2/model_v1/user_index.csv...
Connecting to Persistent User History DB...

========================================
Running CF (ALS)...
Loading Data & CPU FAISS Index...
Cloning FAISS Index to GPU VRAM (for Batch Processing)...
Recommender Engine Ready (Users: 743,401)

CF Done 1890.88 second.
Loading Data & CPU FAISS Index...
Cloning FAISS Index to GPU VRAM (for Batch Processing)...
Recommender Engine Ready (Users: 743,401)

CF Done 1629.25 second.
========================================
Running CBF (NLP)...
Loading Data & CPU FAISS Index...
Cloning FAISS Index to GPU VRAM (for Batch Processing)...
Recommender Engine Ready (Users: 743,401)

Done 3601.28 giây.
ALL DONE



In [5]:
import json
import math
import duckdb
import pandas as pd
import gc
gc.collect()
def get_dcg(hits_ranks):
    dcg = 0.0
    for rank in hits_ranks:
        dcg += 1.0 / math.log2(rank + 1)
    return dcg

def get_idcg(k, actual_len):
    ideal_hits = min(k, actual_len)
    return get_dcg(list(range(1, ideal_hits + 1)))

def calculate_user_metrics(predicted_ids, actual_set, k=10):
    pred_k = predicted_ids[:k]
    hits = 0
    hits_ranks = []
    sum_precisions = 0.0

    for rank, item_id in enumerate(pred_k, start=1):
        if item_id in actual_set:
            hits += 1
            hits_ranks.append(rank)
            sum_precisions += hits / rank

    precision = hits / k
    recall = hits / len(actual_set) if actual_set else 0.0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    average_precision = sum_precisions / min(k, len(actual_set)) if actual_set else 0.0

    idcg = get_idcg(k, len(actual_set))
    ndcg = get_dcg(hits_ranks) / idcg if idcg > 0 else 0.0

    return precision, recall, f1, average_precision, ndcg

def build_user_segments(db_path, test_users):
    print(f"Analyzing user history from {db_path}...")

    # query = """
    #     SELECT
    #         CAST(user_id AS VARCHAR) as uid,
    #         COALESCE(OCTET_LENGTH(seen_books) // 4, 0) as total_seen,
    #         COALESCE(OCTET_LENGTH(rated_books) // 4, 0) as total_rated
    #     FROM user_history
    # """
    query = """
        SELECT
            CAST(user_id AS VARCHAR) as uid,
            COALESCE(len(seen_books), 0) as total_seen,
            COALESCE(len(rated_books), 0) as total_rated
        FROM user_history
    """

    with duckdb.connect(db_path, read_only=True) as conn:
        results = conn.execute(query).fetchall()

    segment_map = {str(uid): "Cold Start" for uid in test_users}
    stats = {"Cold Start": len(test_users), "Medium": 0, "Heavy": 0}

    test_users_set = set(str(u) for u in test_users)

    for row in results:
        uid, total_seen, total_rated = str(row[0]), row[1], row[2]

        if uid not in test_users_set:
            continue

        if total_seen < 20 and total_rated < 5:
            segment = "Cold Start"
        elif total_seen > 100:
            segment = "Heavy"
        else:
            segment = "Medium"

        old_segment = segment_map[uid]
        stats[old_segment] -= 1
        segment_map[uid] = segment
        stats[segment] += 1

    print(f"Test Distribution: Cold Start ({stats['Cold Start']}), Medium ({stats['Medium']}), Heavy ({stats['Heavy']})")
    return segment_map

def evaluate_model_segmented(predictions_file_path, ground_truth_dict, segment_map, k=10):
    """Đánh giá an toàn RAM (Streaming) cho file Standard JSON siêu lớn"""

    metrics_sum = {
        "Cold Start": {"p": 0.0, "r": 0.0, "f1": 0.0, "map": 0.0, "ndcg": 0.0, "count": 0},
        "Medium":     {"p": 0.0, "r": 0.0, "f1": 0.0, "map": 0.0, "ndcg": 0.0, "count": 0},
        "Heavy":      {"p": 0.0, "r": 0.0, "f1": 0.0, "map": 0.0, "ndcg": 0.0, "count": 0},
        "Overall":    {"p": 0.0, "r": 0.0, "f1": 0.0, "map": 0.0, "ndcg": 0.0, "count": 0}
    }

    print(f"⏳ Đang chấm điểm file JSON siêu lớn (Chế độ Streaming an toàn): {predictions_file_path}...")

    # MỞ FILE ĐỌC TỪNG DÒNG (KHÔNG DÙNG json.load Toàn Bộ)
    with open(predictions_file_path, 'r', encoding='utf-8') as f:
        for line_no, line in enumerate(f, 1):
            line = line.strip()

            # Bỏ qua các dòng chỉ chứa dấu mở { hoặc đóng } của toàn bộ file
            if line == "{" or line == "}":
                continue
            if not line:
                continue

            # Xóa dấu phẩy ở cuối dòng (nếu có) để tạo thành chuỗi JSON hợp lệ
            if line.endswith(','):
                line = line[:-1]

            # Lúc này line có dạng: "user_id": [{"item_id": "123", ...}]
            # Ta bọc nó lại thành một Object Dictionary hoàn chỉnh để ép json.loads() dịch
            json_str = "{" + line + "}"

            try:
                record = json.loads(json_str)
            except Exception as e:
                # Nếu có một dòng format bị nát, in log và bỏ qua chứ không sập máy
                if line_no < 5: # Chỉ in vài dòng đầu tránh rác màn hình
                    print(f"⚠️ Bỏ qua dòng {line_no} do lỗi: {e}")
                continue

            # record giờ là 1 Dictionary nhỏ giọt chỉ chứa ĐÚNG 1 USER
            for uid, recommendations in record.items():
                if uid not in ground_truth_dict or uid not in segment_map:
                    continue

                actual_set = set(ground_truth_dict[uid])
                if not actual_set:
                    continue

                predicted_ids = [item['item_id'] for item in recommendations]
                p, r, f1, ap, ndcg = calculate_user_metrics(predicted_ids, actual_set, k)

                seg = segment_map[uid]

                metrics_sum[seg]["p"] += p
                metrics_sum[seg]["r"] += r
                metrics_sum[seg]["f1"] += f1
                metrics_sum[seg]["map"] += ap
                metrics_sum[seg]["ndcg"] += ndcg
                metrics_sum[seg]["count"] += 1

                metrics_sum["Overall"]["p"] += p
                metrics_sum["Overall"]["r"] += r
                metrics_sum["Overall"]["f1"] += f1
                metrics_sum["Overall"]["map"] += ap
                metrics_sum["Overall"]["ndcg"] += ndcg
                metrics_sum["Overall"]["count"] += 1

    # Tính trung bình cộng
    final_results = {}
    for seg, data in metrics_sum.items():
        count = data["count"]
        if count == 0:
            final_results[seg] = [0.0, 0.0, 0.0, 0.0, 0.0]
        else:
            final_results[seg] = [
                data["p"] / count, data["r"] / count,
                data["f1"] / count, data["map"] / count,
                data["ndcg"] / count
            ]

    return final_results

def main():
    print("STARTING SEGMENTED EVALUATION")



    with open(ground_truth_file, 'r', encoding='utf-8') as f:
        ground_truth = json.load(f)
    test_users = list(ground_truth.keys())

    segment_map = build_user_segments(history_db, test_users)

    models_to_eval = {
        "ALS (Collaborative 384 factor)": f"{_eval_dir}/cf_test_predictions.json",
        "ALS (Collaborative 32 factor)": f"{_eval_dir}/cf_test_predictions_32.json",
        "SBERT_W1 (Content-Based)": cbf_prediction_v2_w1,
        "SBERT_W2 (Content-Based)": cbf_prediction_v2_w2,
        # "SBERT_W1 (Content-Based)": f"{_eval_dir}/cbf_test_predictions.json",
        # "SBERT_W2 (Content-Based)": f"{_eval_dir}/cbf_test_predictions_v2.json"
    }

    k_val = 10
    all_matrices = []

    for model_name, path in models_to_eval.items():
        try:
            results = evaluate_model_segmented(path, ground_truth, segment_map, k=k_val)

            df = pd.DataFrame.from_dict(
                results,
                orient='index',
                columns=[f'Precision@{k_val}', f'Recall@{k_val}', f'F1@{k_val}', f'MAP@{k_val}', f'NDCG@{k_val}']
            )
            df.insert(0, 'Model', model_name)
            df.index.name = 'User Group'
            all_matrices.append(df)

        except FileNotFoundError:
            print(f"File not found: {path}. Skipping.")

    if all_matrices:
        final_df = pd.concat(all_matrices).reset_index()
        final_df = final_df.set_index(['User Group', 'Model']).sort_index(level=0)

        df_formatted = final_df.map(lambda x: f"{x * 100:.3f}%" if isinstance(x, float) else x)

        print("\n" + "="*80)
        print("SEGMENTED EVALUATION MATRIX")
        print("="*80)
        print(df_formatted.to_string())
        print("="*80)

        final_df.to_csv(f"{_eval_dir}/segmented_evaluation_matrix.csv")
        print(f"\nMatrix saved to:{_eval_dir} /segmented_evaluation_matrix.csv")

if __name__ == "__main__":
    main()


STARTING SEGMENTED EVALUATION
Analyzing user history from /content/drive/My Drive/Thesis/book_recsys/data/processed_2/model_v1/user_history.db...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Test Distribution: Cold Start (12204), Medium (352309), Heavy (378888)
⏳ Đang chấm điểm file JSON siêu lớn (Chế độ Streaming an toàn): /content/drive/My Drive/Thesis/book_recsys/data/processed_2/eval_v1/cf_test_predictions.json...
⏳ Đang chấm điểm file JSON siêu lớn (Chế độ Streaming an toàn): /content/drive/My Drive/Thesis/book_recsys/data/processed_2/eval_v1/cf_test_predictions_32.json...
⏳ Đang chấm điểm file JSON siêu lớn (Chế độ Streaming an toàn): /content/drive/My Drive/Thesis/book_recsys/data/processed_2/eval_v1/cbf_test_predictions_v2_w1.json...
⏳ Đang chấm điểm file JSON siêu lớn (Chế độ Streaming an toàn): /content/drive/My Drive/Thesis/book_recsys/data/processed_2/eval_v1/cbf_test_predictions_v2_w2.json...

SEGMENTED EVALUATION MATRIX
                                          Precision@10 Recall@10   F1@10  MAP@10  NDCG@10
User Group Model                                                                         
Cold Start ALS (Collaborative 32 factor)        1.084%    4.759

In [9]:
import json
import math
import duckdb
import pandas as pd
import gc
import os

gc.collect()

def get_dcg(hits_ranks):
    dcg = 0.0
    for rank in hits_ranks:
        dcg += 1.0 / math.log2(rank + 1)
    return dcg

def get_idcg(k, actual_len):
    ideal_hits = min(k, actual_len)
    return get_dcg(list(range(1, ideal_hits + 1)))

def calculate_user_metrics(predicted_ids, actual_set, k=10):
    pred_k = predicted_ids[:k]
    hits = 0
    hits_ranks = []
    sum_precisions = 0.0

    for rank, item_id in enumerate(pred_k, start=1):
        if item_id in actual_set:
            hits += 1
            hits_ranks.append(rank)
            sum_precisions += hits / rank

    precision = hits / k
    recall = hits / len(actual_set) if actual_set else 0.0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    average_precision = sum_precisions / min(k, len(actual_set)) if actual_set else 0.0

    idcg = get_idcg(k, len(actual_set))
    ndcg = get_dcg(hits_ranks) / idcg if idcg > 0 else 0.0

    return precision, recall, f1, average_precision, ndcg

def build_user_segments(db_path, test_users):
    print(f"Analyzing user history from {db_path}...")

    query = """
        SELECT
            CAST(user_id AS VARCHAR) as uid,
            COALESCE(len(seen_books), 0) as total_seen,
            COALESCE(len(rated_books), 0) as total_rated
        FROM user_history
    """

    with duckdb.connect(db_path, read_only=True) as conn:
        results = conn.execute(query).fetchall()

    segment_map = {str(uid): "Cold Start" for uid in test_users}
    stats = {"Cold Start": len(test_users), "Medium": 0, "Heavy": 0}

    test_users_set = set(str(u) for u in test_users)

    for row in results:
        uid, total_seen, total_rated = str(row[0]), row[1], row[2]

        if uid not in test_users_set:
            continue

        if total_seen < 20 and total_rated < 5:
            segment = "Cold Start"
        elif total_seen > 100:
            segment = "Heavy"
        else:
            segment = "Medium"

        old_segment = segment_map[uid]
        stats[old_segment] -= 1
        segment_map[uid] = segment
        stats[segment] += 1

    print(f"Test Distribution: Cold Start ({stats['Cold Start']}), Medium ({stats['Medium']}), Heavy ({stats['Heavy']})")
    return segment_map

def evaluate_model_segmented(predictions_file_path, ground_truth_dict, segment_map, k=10):
    metrics_sum = {
        "Cold Start": {"p": 0.0, "r": 0.0, "f1": 0.0, "map": 0.0, "ndcg": 0.0, "count": 0},
        "Medium":     {"p": 0.0, "r": 0.0, "f1": 0.0, "map": 0.0, "ndcg": 0.0, "count": 0},
        "Heavy":      {"p": 0.0, "r": 0.0, "f1": 0.0, "map": 0.0, "ndcg": 0.0, "count": 0},
        "Overall":    {"p": 0.0, "r": 0.0, "f1": 0.0, "map": 0.0, "ndcg": 0.0, "count": 0}
    }

    print(f"Evaluating (streaming mode): {predictions_file_path}...")

    with open(predictions_file_path, 'r', encoding='utf-8') as f:
        for line_no, line in enumerate(f, 1):
            line = line.strip()

            if line == "{" or line == "}":
                continue
            if not line:
                continue

            if line.endswith(','):
                line = line[:-1]

            json_str = "{" + line + "}"

            try:
                record = json.loads(json_str)
            except Exception as e:
                if line_no < 5:
                    print(f"Skipping line {line_no}: {e}")
                continue

            for uid, recommendations in record.items():
                if uid not in ground_truth_dict or uid not in segment_map:
                    continue

                actual_set = set(ground_truth_dict[uid])
                if not actual_set:
                    continue

                predicted_ids = [item['item_id'] for item in recommendations]
                p, r, f1, ap, ndcg = calculate_user_metrics(predicted_ids, actual_set, k)

                seg = segment_map[uid]

                metrics_sum[seg]["p"] += p
                metrics_sum[seg]["r"] += r
                metrics_sum[seg]["f1"] += f1
                metrics_sum[seg]["map"] += ap
                metrics_sum[seg]["ndcg"] += ndcg
                metrics_sum[seg]["count"] += 1

                metrics_sum["Overall"]["p"] += p
                metrics_sum["Overall"]["r"] += r
                metrics_sum["Overall"]["f1"] += f1
                metrics_sum["Overall"]["map"] += ap
                metrics_sum["Overall"]["ndcg"] += ndcg
                metrics_sum["Overall"]["count"] += 1

    final_results = {}
    for seg, data in metrics_sum.items():
        count = data["count"]
        if count == 0:
            final_results[seg] = [0.0, 0.0, 0.0, 0.0, 0.0]
        else:
            final_results[seg] = [
                data["p"] / count, data["r"] / count,
                data["f1"] / count, data["map"] / count,
                data["ndcg"] / count
            ]

    return final_results

def main():
    print("STARTING SEGMENTED EVALUATION (MULTI-WEIGHT HYBRID)")

    with open(ground_truth_file, 'r', encoding='utf-8') as f:
        ground_truth = json.load(f)
    test_users = list(ground_truth.keys())

    segment_map = build_user_segments(history_db, test_users)

    # --- Base models ---
    models_to_eval = {
        # "ALS 384 (CF)":             f"{_eval_dir}/cf_test_predictions.json",
        # "ALS 32 (CF)":              f"{_eval_dir}/cf_test_predictions_32.json",
        # "SBERT_W1 (Content-Based)": f"{_eval_dir}/cbf_test_predictions.json",
        # "SBERT_W2 (Content-Based)": f"{_eval_dir}/cbf_test_predictions_v2.json",
    }

    # --- Hybrid weight configurations ---
    hybrid_dir = f"{_eval_dir}/hybrid"
    weight_pairs = [
        (1.0, 0.0),
        (0.8, 0.2),
        (0.6, 0.4),
        (0.4, 0.6),
        (0.2, 0.8),
        (0.0, 1.0),
    ]

    for w_cf, w_cbf in weight_pairs:
        filename = f"hybrid_{w_cf:.1f}_{w_cbf:.1f}_v2_w1.json"
        filepath = os.path.join(hybrid_dir, filename)
        label = f"Hybrid (CF={w_cf:.1f}, CBF={w_cbf:.1f})"
        models_to_eval[label] = filepath

    k_val = 10
    all_matrices = []

    for model_name, path in models_to_eval.items():
        if not os.path.exists(path):
            print(f"File not found: {path}. Skipping.")
            continue

        try:
            results = evaluate_model_segmented(path, ground_truth, segment_map, k=k_val)

            df = pd.DataFrame.from_dict(
                results,
                orient='index',
                columns=[f'Precision@{k_val}', f'Recall@{k_val}', f'F1@{k_val}', f'MAP@{k_val}', f'NDCG@{k_val}']
            )
            df.insert(0, 'Model', model_name)
            df.index.name = 'User Group'
            all_matrices.append(df)

        except Exception as e:
            print(f"Error evaluating {model_name}: {e}. Skipping.")

    if all_matrices:
        final_df = pd.concat(all_matrices).reset_index()
        final_df = final_df.set_index(['User Group', 'Model']).sort_index(level=0)

        df_formatted = final_df.map(lambda x: f"{x * 100:.3f}%" if isinstance(x, float) else x)

        print("\n" + "=" * 100)
        print("SEGMENTED EVALUATION MATRIX (BASE MODELS + HYBRID SENSITIVITY)")
        print("=" * 100)
        print(df_formatted.to_string())
        print("=" * 100)

        output_csv = f"{_eval_dir}/segmented_evaluation_with_hybrid_v2.csv"
        final_df.to_csv(output_csv)
        print(f"\nMatrix saved to: {output_csv}")

if __name__ == "__main__":
    main()


STARTING SEGMENTED EVALUATION (MULTI-WEIGHT HYBRID)
Analyzing user history from /content/drive/My Drive/Thesis/book_recsys/data/processed_2/model_v1/user_history.db...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Test Distribution: Cold Start (12204), Medium (352309), Heavy (378888)
Evaluating (streaming mode): /content/drive/My Drive/Thesis/book_recsys/data/processed_2/eval_v1/hybrid/hybrid_1.0_0.0_v2_w1.json...
Evaluating (streaming mode): /content/drive/My Drive/Thesis/book_recsys/data/processed_2/eval_v1/hybrid/hybrid_0.8_0.2_v2_w1.json...
Evaluating (streaming mode): /content/drive/My Drive/Thesis/book_recsys/data/processed_2/eval_v1/hybrid/hybrid_0.6_0.4_v2_w1.json...
Evaluating (streaming mode): /content/drive/My Drive/Thesis/book_recsys/data/processed_2/eval_v1/hybrid/hybrid_0.4_0.6_v2_w1.json...
Evaluating (streaming mode): /content/drive/My Drive/Thesis/book_recsys/data/processed_2/eval_v1/hybrid/hybrid_0.2_0.8_v2_w1.json...
Evaluating (streaming mode): /content/drive/My Drive/Thesis/book_recsys/data/processed_2/eval_v1/hybrid/hybrid_0.0_1.0_v2_w1.json...

SEGMENTED EVALUATION MATRIX (BASE MODELS + HYBRID SENSITIVITY)
                                    Precision@10 Recall@10   F1@10 

In [ ]:
import gc
try:
  del cb_recommend
  del cf_recommend
except:
  pass
gc.collect()
cb_recommend = VectorRecommender(
    book_processor= book_proc,
    user_processor= user_proc,
    faiss_index_path=sbert_item_faiss_index,
    user_vectors_path = sbert_user_matrix,

)
generate_recommendations_from_ground_truth(
    recommender = cb_recommend,
    input_csv = f'{_data_dir}/eval_v1/ground_true_test.csv',
    output_csv= f'{_data_dir}/eval_v1/cb_prediction.csv',
    k_top = 10
)
# del cb_recommend
# gc.collect()

cf_recommend = VectorRecommender(
    book_processor= book_proc,
    user_processor= user_proc,
    faiss_index_path=ials_item_faiss_index,
    user_vectors_path = ials_user_matrix,

)
generate_recommendations_from_ground_truth(
    recommender = cb_recommend,
    input_csv = f'{_data_dir}/eval_v1/ground_true_test.csv',
    output_csv =f'{_data_dir}/eval_v1/cf_prediction.csv',
    k_top = 10
)


In [ ]:
import pandas as pd
import numpy as np
def evaluate_recommender(gt_csv: str, pred_csv: str, k: int = 10):
    """
    Tính toán Precision, Recall, F1, MAP và NDCG.

    Args:
        gt_csv: Đường dẫn file ground truth (user_id, books_ids)
        pred_csv: Đường dẫn file dự đoán (user_id, books_ids)
        k: Ngưỡng tính toán (ví dụ @10)
    """
    print(f"📊 Đang tải dữ liệu để đánh giá @{k}...")

    # Đọc dữ liệu
    df_gt = pd.read_csv(gt_csv)
    df_pred = pd.read_csv(pred_csv)

    # Merge để đảm bảo so sánh đúng user_id
    df = pd.merge(df_gt, df_pred, on='user_id', suffixes=('_gt', '_pred'))

    metrics = {
        'precision': [],
        'recall': [],
        'ap': [],      # Dùng cho MAP
        'ndcg': []
    }

    for _, row in df.iterrows():
        # Chuyển chuỗi space-separated thành list/set
        true_items = set(str(row['books_ids_gt']).split())
        pred_items = str(row['books_ids_pred']).split()[:k]

        if not true_items or not pred_items:
            for m in metrics: metrics[m].append(0)
            continue

        hits = 0
        ap_sum = 0
        dcg = 0
        idcg = 0

        # Tính toán từng vị trí trong danh sách dự đoán
        for i, item in enumerate(pred_items):
            if item in true_items:
                hits += 1
                # Phần tử cho Average Precision
                ap_sum += hits / (i + 1)
                # Phần tử cho DCG
                dcg += 1 / np.log2(i + 2)

        # Tính IDCG (Ideal DCG - khi các item đúng nằm ở đầu)
        for i in range(min(len(true_items), k)):
            idcg += 1 / np.log2(i + 2)

        # Precision & Recall
        prec = hits / k
        rec = hits / len(true_items)

        # Average Precision cho user này
        ap = ap_sum / min(len(true_items), k)

        # NDCG
        ndcg = dcg / idcg if idcg > 0 else 0

        metrics['precision'].append(prec)
        metrics['recall'].append(rec)
        metrics['ap'].append(ap)
        metrics['ndcg'].append(ndcg)

    # Tính trung bình tất cả user
    avg_precision = np.mean(metrics['precision'])
    avg_recall = np.mean(metrics['recall'])
    avg_map = np.mean(metrics['ap'])
    avg_ndcg = np.mean(metrics['ndcg'])

    # Tính F1 từ Precision và Recall trung bình
    f1 = (2 * avg_precision * avg_recall / (avg_precision + avg_recall)) if (avg_precision + avg_recall) > 0 else 0

    print(f"\n--- Kết quả đánh giá @{k} ---")
    print(f"Precision: {avg_precision:.4f}")
    print(f"Recall:    {avg_recall:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print(f"MAP:       {avg_map:.4f}")
    print(f"NDCG:      {avg_ndcg:.4f}")

    return {
        'precision': avg_precision,
        'recall': avg_recall,
        'f1': f1,
        'map': avg_map,
        'ndcg': avg_ndcg
    }

In [ ]:
import pandas as pd
import numpy as np

def evaluate_recommender(gt_csv: str, pred_csv: str, k: int = 10):
    print(f"📊 Đang tải dữ liệu để đánh giá @{k}...")

    df_gt = pd.read_csv(gt_csv)
    df_pred = pd.read_csv(pred_csv)
    df = pd.merge(df_gt, df_pred, on='user_id', suffixes=('_gt', '_pred'))

    metrics = {'precision': [], 'recall': [], 'ap': [], 'ndcg': []}

    # [TỐI ƯU HÓA HIỆU NĂNG]: Dùng zip thay cho iterrows()
    gt_list = df['books_ids_gt'].values
    pred_list = df['books_ids_pred'].values

    for gt_str, pred_str in zip(gt_list, pred_list):
        # Chặn lỗi NaN (Người dùng không có dữ liệu thật hoặc không có dự đoán)
        if pd.isna(gt_str) or pd.isna(pred_str):
            for m in metrics: metrics[m].append(0)
            continue

        true_items = set(str(gt_str).split())
        pred_items = str(pred_str).split()[:k]

        if not true_items or not pred_items:
            for m in metrics: metrics[m].append(0)
            continue

        hits = 0
        ap_sum = 0.0
        dcg = 0.0
        idcg = 0.0

        for i, item in enumerate(pred_items):
            if item in true_items:
                hits += 1
                ap_sum += hits / (i + 1)
                dcg += 1.0 / np.log2(i + 2)

        # Tính IDCG
        num_ideal_hits = min(len(true_items), k)
        for i in range(num_ideal_hits):
            idcg += 1.0 / np.log2(i + 2)

        prec = hits / k
        rec = hits / len(true_items)
        ap = ap_sum / num_ideal_hits if num_ideal_hits > 0 else 0.0
        ndcg = dcg / idcg if idcg > 0 else 0.0

        metrics['precision'].append(prec)
        metrics['recall'].append(rec)
        metrics['ap'].append(ap)
        metrics['ndcg'].append(ndcg)

    # Tính trung bình
    avg_precision = np.mean(metrics['precision'])
    avg_recall = np.mean(metrics['recall'])
    avg_map = np.mean(metrics['ap'])
    avg_ndcg = np.mean(metrics['ndcg'])

    # Tính F1
    sum_pr = avg_precision + avg_recall
    f1 = (2 * avg_precision * avg_recall / sum_pr) if sum_pr > 0 else 0

    print(f"\n--- Kết quả đánh giá @{k} ---")
    print(f"Precision: {avg_precision:.4f}")
    print(f"Recall:    {avg_recall:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print(f"MAP:       {avg_map:.4f}")
    print(f"NDCG:      {avg_ndcg:.4f}")

    return {
        'precision': avg_precision,
        'recall': avg_recall,
        'f1': f1,
        'map': avg_map,
        'ndcg': avg_ndcg
    }

In [ ]:
try:
  del cb_recommend
  del cf_recommend
except:
  pass
import gc
gc.collect()
g_true = f"{_data_dir}/eval_v1/ground_true_test.csv"
cb_pred = f"{_data_dir}/eval_v1/cb_prediction.csv"
cf_pred = f"{_data_dir}/eval_v1/cf_prediction.csv"
print(evaluate_recommender(gt_csv=g_true, pred_csv=cb_pred))
gc.collect()
print(evaluate_recommender(gt_csv=g_true,pred_csv=cf_pred))

# evaluator = RecommenderEvaluator (g_true)
# evaluator.evaluate(cb_pred)
# evaluator.evaluate(cf_pred)

📊 Đang tải dữ liệu để đánh giá @10...

--- Kết quả đánh giá @10 ---
Precision: 0.0002
Recall:    0.0004
F1-Score:  0.0003
MAP:       0.0002
NDCG:      0.0003
{'precision': np.float64(0.00019501591491343837), 'recall': np.float64(0.00039611788909382566), 'f1': np.float64(0.00026135975317785436), 'map': np.float64(0.00016299689813272018), 'ndcg': np.float64(0.0003308858084311252)}
📊 Đang tải dữ liệu để đánh giá @10...

--- Kết quả đánh giá @10 ---
Precision: 0.0002
Recall:    0.0004
F1-Score:  0.0003
MAP:       0.0002
NDCG:      0.0003
{'precision': np.float64(0.00019501591491343837), 'recall': np.float64(0.00039611788909382566), 'f1': np.float64(0.00026135975317785436), 'map': np.float64(0.00016299689813272018), 'ndcg': np.float64(0.0003308858084311252)}


In [ ]:
import pandas as pd
# Assuming 'evaluate_comprehensive_metrics' and 'ground_truth' are loaded in memory

print("Evaluating all model predictions against Ground Truth...")

leaderboard_data = []

for model_name in EXPERIMENTS.keys():
    pred_filepath = os.path.join(OUTPUT_DIR, f"{model_name}_preds.json")

    if not os.path.exists(pred_filepath):
        print(f"Missing predictions for {model_name}. Skipping evaluation.")
        continue

    # Load Predictions from Disk
    with open(pred_filepath, 'r', encoding='utf-8') as f:
        loaded_predictions = json.load(f)

    # Calculate Metrics
    results = evaluate_comprehensive_metrics(loaded_predictions, ground_truth, K_RECOMMENDATIONS)

    # Format data for Pandas
    leaderboard_data.append({
        "Model": model_name,
        "Precision": results[f'Precision@{K_RECOMMENDATIONS}'] * 100,
        "Recall": results[f'Recall@{K_RECOMMENDATIONS}'] * 100,
        "F1-Score": results[f'F1@{K_RECOMMENDATIONS}'] * 100,
        "MAP": results[f'MAP@{K_RECOMMENDATIONS}'] * 100,
        "NDCG": results[f'NDCG@{K_RECOMMENDATIONS}'] * 100
    })

# Generate the Final Leaderboard
df_leaderboard = pd.DataFrame(leaderboard_data)

# Sort by NDCG (Industry standard for ranking models)
df_leaderboard = df_leaderboard.sort_values(by="NDCG", ascending=False).reset_index(drop=True)

# Display a beautiful academic table
print("\n" + "="*80)
print(f"🏆 MODEL LEADERBOARD (Top {K_RECOMMENDATIONS} Recommendations)")
print("="*80)

# Format floats to 2 decimal places with '%'
pd.options.display.float_format = '{:.2f}%'.format
display(df_leaderboard)